# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Packages installed.')


# Data Anonymization & Privacy

Protecting participant privacy is a core ethical requirement in social science. This notebook demonstrates techniques for de-identifying data, including hashing, k-anonymity, and differential privacy concepts.

## Key Concepts
1.  **Direct Identifiers**: Names, SSNs, Emails (Must be removed/hashed).
2.  **Quasi-Identifiers (QI)**: Age, Zip Code, Gender (Can be combined to re-identify).
3.  **K-Anonymity**: Ensuring any record is indistinguishable from at least *k-1* other records.

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

## 1. Create Sensitive Dataset
We simulate a dataset containing medical records.

In [ ]:
data = {
    'Name': ['Alice Smith', 'Bob Jones', 'Charlie Brown', 'David White', 'Eva Green', 'Frank Black'],
    'Age': [25, 34, 28, 25, 34, 45],
    'ZipCode': ['10001', '10002', '10001', '10001', '10002', '90210'],
    'Diagnosis': ['Flu', 'Cancer', 'Flu', 'Diabetes', 'Cancer', 'Heart Disease']
}
df = pd.DataFrame(data)
print("Original Sensitive Data:")
print(df)

## 2. Hashing Direct Identifiers
Names should never be shared. We use SHA-256 with a 'salt' to prevent rainbow table attacks.

In [ ]:
SALT = "ResearchProject2023_"

def secure_hash(val):
    # Combine value with secret salt
    salted_val = SALT + val
    return hashlib.sha256(salted_val.encode()).hexdigest()

df['ID'] = df['Name'].apply(secure_hash)
# Drop original name
df_anon = df.drop(columns=['Name'])

# Reorder columns
cols = ['ID'] + [c for c in df_anon.columns if c != 'ID']
df_anon = df_anon[cols]

print("Hashed Data:")
print(df_anon.head())

## 3. K-Anonymity
Even without names, a combination of Age + Zip Code could identify someone (e.g., the only 45-year-old in Zip 90210). 

**Definition**: A dataset is k-anonymous if every combination of Quasi-Identifiers appears at least *k* times.

Let's check k for our current dataset using [Age, ZipCode].

In [ ]:
def get_k_anonymity(df, qi_cols):
    # Count occurrences of each QI combination
    counts = df.groupby(qi_cols).size().reset_index(name='count')
    return counts['count'].min(), counts

qi = ['Age', 'ZipCode']
min_k, groups = get_k_anonymity(df_anon, qi)

print(f"Current K-Anonymity: {min_k}")
print("Groups with low counts:")
print(groups[groups['count'] < 2])

## 4. Generalization
To improve K, we generalize the data (make it less specific). We can bin ages and truncate zip codes.

In [ ]:
# 1. Generalize Age (10-year bins)
# Floor division to nearest 10
df_anon['Age_Bin'] = (df_anon['Age'] // 10) * 10
df_anon['Age_Bin'] = df_anon['Age_Bin'].astype(str) + "-" + (df_anon['Age_Bin'] + 10).astype(str)

# 2. Generalize Zip (First 3 digits)
df_anon['Zip_Short'] = df_anon['ZipCode'].str[:3] + "**"

# Check K again with new QIs
new_qi = ['Age_Bin', 'Zip_Short']
min_k_new, groups_new = get_k_anonymity(df_anon, new_qi)

print("Generalized Data:")
print(df_anon[['ID', 'Age_Bin', 'Zip_Short', 'Diagnosis']].head())
print(f"\nNew K-Anonymity: {min_k_new}")

## 5. Differential Privacy (Concept)
For aggregate statistics (e.g., "What is the average age of cancer patients?"), we can add random noise so the output doesn't reveal if any specific individual was in the dataset.

We use the **Laplace Mechanism**.

In [ ]:
def add_laplace_noise(true_value, sensitivity, epsilon):
    """
    true_value: The actual statistic
    sensitivity: How much one person can change the result (e.g., 1 for count)
    epsilon: Privacy budget (lower = more privacy, more noise)
    """
    noise = np.random.laplace(0, sensitivity / epsilon)
    return true_value + noise

# Example: Count number of Cancer patients
true_count = len(df[df['Diagnosis'] == 'Cancer'])

# Add noise (epsilon=1.0 is common)
private_count = add_laplace_noise(true_count, sensitivity=1, epsilon=1.0)

print(f"True Count: {true_count}")
print(f"Differentially Private Count: {private_count:.2f}")
print("Note: The private count is not an integer, masking the exact number.")